
# Student Placement Analysis: EDA, Modeling, and Deployment Prep

Notebook ini memuat eksplorasi data, feature engineering, train-test split 80:20, dan perbandingan minimal tiga model untuk:
1. **Klasifikasi**: `placement_status`
2. **Regresi**: `salary_lpa`

Catatan penting: pada data target, `salary_lpa = 0` untuk mahasiswa yang `Not Placed`. Karena itu, model regresi yang paling masuk akal dibangun pada subset `Placed` agar tidak belajar pola trivial dari label penempatan.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from feature_engineering import load_datasets, StudentFeatureEngineer


In [ ]:

# Load and merge data
data = load_datasets("A.csv", "A_targets.csv")
data = StudentFeatureEngineer().fit_transform(data)

data.head()


## 1) Data overview

In [ ]:

print("Shape:", data.shape)
display(data.info())
display(data.isna().sum().sort_values(ascending=False).head(10))
display(data['placement_status'].value_counts())
display(data['salary_lpa'].describe())


## 2) Exploratory Data Analysis

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data['placement_status'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Placement Outcome Distribution')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')

sns.histplot(data['salary_lpa'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('Salary Distribution')

plt.tight_layout()
plt.show()


In [ ]:

# Correlation on numeric columns
num_cols = data.select_dtypes(include=np.number).columns
corr = data[num_cols].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

placement_numeric = data.copy()
placement_numeric['placement_binary'] = (placement_numeric['placement_status'] == 'Placed').astype(int)
display(
    placement_numeric.select_dtypes(include=np.number).corr(numeric_only=True)['placement_binary']
    .sort_values(ascending=False)
    .head(12)
)



### Interpretasi singkat

Fitur yang biasanya paling kuat untuk penempatan dan salary pada data ini adalah:
- **cgpa**
- **tenth_percentage** dan **twelfth_percentage**
- **internships_completed**
- **coding_skill_rating**
- **projects_completed**
- **hackathons_participated**
- **aptitude_skill_rating**

Fitur gaya hidup seperti `sleep_hours` dan `stress_level` tetap dipertahankan karena bisa memberi sinyal tambahan, walau korelasinya biasanya lebih kecil.


## 3) Feature engineering

In [ ]:

# Contoh feature engineering
feature_preview = data[['cgpa','tenth_percentage','twelfth_percentage',
                        'coding_skill_rating','communication_skill_rating','aptitude_skill_rating',
                        'projects_completed','internships_completed','hackathons_participated','certifications_count',
                        'sleep_hours','stress_level','academic_average','technical_average','activity_score','lifestyle_balance']]
feature_preview.head()


## 4) Train-test split (80:20)

In [ ]:

# Classification dataset
X_cls = data.drop(columns=['Student_ID', 'placement_status', 'salary_lpa'])
y_cls = data['placement_status']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Regression dataset: only placed students
reg_data = data[data['placement_status'] == 'Placed'].copy()
X_reg = reg_data.drop(columns=['Student_ID', 'placement_status', 'salary_lpa'])
y_reg = reg_data['salary_lpa']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("Classification train/test:", X_train_c.shape, X_test_c.shape)
print("Regression train/test:", X_train_r.shape, X_test_r.shape)


## 5) Modeling: Classification

In [ ]:

num_cols_cls = X_cls.select_dtypes(include=np.number).columns.tolist()
cat_cols_cls = X_cls.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_cls = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols_cls),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols_cls),
    ]
)

cls_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced_subsample'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

cls_results = []
for name, model in cls_models.items():
    pipe = Pipeline([('preprocess', preprocessor_cls), ('model', model)])
    pipe.fit(X_train_c, y_train_c)
    pred = pipe.predict(X_test_c)
    proba = pipe.predict_proba(X_test_c)[:, 1]
    cls_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_c, pred),
        'Precision': precision_score(y_test_c, pred, pos_label='Placed'),
        'Recall': recall_score(y_test_c, pred, pos_label='Placed'),
        'F1': f1_score(y_test_c, pred, pos_label='Placed'),
        'ROC AUC': roc_auc_score((y_test_c == 'Placed').astype(int), proba),
    })

cls_results_df = pd.DataFrame(cls_results).sort_values('F1', ascending=False)
display(cls_results_df)



**Interpretasi klasifikasi**:  
F1-score dipilih sebagai metrik utama karena kelas `Not Placed` relatif minoritas. Pada eksperimen ini, model berbasis pohon biasanya unggul dalam menangkap interaksi non-linear antar fitur akademik, teknis, dan aktivitas.


## 6) Modeling: Regression

In [ ]:

num_cols_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
cat_cols_reg = X_reg.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols_reg),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols_reg),
    ]
)

reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=300, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42),
}

reg_results = []
for name, model in reg_models.items():
    pipe = Pipeline([('preprocess', preprocessor_reg), ('model', model)])
    pipe.fit(X_train_r, y_train_r)
    pred = pipe.predict(X_test_r)
    reg_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test_r, pred),
        'RMSE': mean_squared_error(y_test_r, pred) ** 0.5,
        'R2': r2_score(y_test_r, pred),
    })

reg_results_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
display(reg_results_df)



**Interpretasi regresi**:  
R2 digunakan untuk menilai seberapa baik variasi salary dijelaskan oleh fitur-fitur input. MAE dan RMSE membantu membaca besarnya error dalam satuan `lpa`.



## 8) Hasil eksperimen model

### Klasifikasi (`placement_status`)
| Model | Accuracy | Precision | Recall | F1 | ROC AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.824 | 0.971 | 0.820 | 0.889 | 0.911 |
| Random Forest | 0.888 | 0.895 | 0.985 | 0.938 | 0.898 |
| Gradient Boosting | 0.882 | 0.910 | 0.957 | 0.933 | 0.909 |

### Regresi (`salary_lpa`, hanya data *Placed*)
| Model | MAE | RMSE | R2 |
|---|---:|---:|---:|
| Linear Regression | 1.132 | 1.408 | 0.771 |
| Random Forest Regressor | 1.172 | 1.476 | 0.749 |
| Gradient Boosting Regressor | 1.106 | 1.398 | 0.774 |

**Model terbaik yang dipakai di deployment**:
- Klasifikasi: **Random Forest**
- Regresi: **Gradient Boosting Regressor**


## 7) Conclusion


Kesimpulan umum:
- Model klasifikasi terbaik cenderung memberikan performa kuat pada fitur akademik, teknis, dan aktivitas.
- Model regresi terbaik untuk salary biasanya bekerja lebih baik pada subset `Placed`.
- Pipeline produksi sebaiknya memakai preprocessing + model dalam satu alur agar menghindari data leakage.
